In [ ]:
# install the shared modeling libraries
!pip install -U scikit-learn xgboost pandas numpy tqdm


# Feature extraction, egemaps and compare2016 datasets

this notebook was originally used in google colab

In [ ]:
# mounting drive and getting data

# connect colab to google drive
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

# data handling
import numpy as np
import pandas as pd

from tqdm import tqdm

# preprocessing and pipelines
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

# classifiers
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)

from xgboost import XGBClassifier

# define project folders
PROJECT = Path("/content/drive/MyDrive/asr project")
FEATURES_ROOT = PROJECT / "features"
SPLIT_ROOT = PROJECT / "splits"

# stop early when required folders are missing
assert PROJECT.exists(), f"Project folder not found: {PROJECT}"
assert FEATURES_ROOT.exists(), f"Features folder not found: {FEATURES_ROOT}"
assert SPLIT_ROOT.exists(), f"Split folder not found: {SPLIT_ROOT}"

print("Project:", PROJECT)
print("Features root:", FEATURES_ROOT)
print("Split root:", SPLIT_ROOT)


In [ ]:
# 2 choose the acoustic feature sets

# both notebooks used the same pipeline and only changed this configuration
FEATURE_CONFIGS = {
    "egemaps": {
        "feature_csv": (
            FEATURES_ROOT
            / "explainable_acoustic_patient_only"
            / "egemaps_features.csv"
        ),
        "result_prefix": "interpretable_egemaps",
    },
    "compare2016": {
        "feature_csv": (
            FEATURES_ROOT
            / "explainable_acoustic_patient_only"
            / "compare2016_features.csv"
        ),
        "result_prefix": "interpretable_compare2016",
    },
}

# use ["egemaps"], ["compare2016"], or both
FEATURES_TO_RUN = ["egemaps", "compare2016"]

# validate the selected names and files
unknown_features = [
    name for name in FEATURES_TO_RUN
    if name not in FEATURE_CONFIGS
]
assert not unknown_features, f"Unknown feature selections: {unknown_features}"

for feature_name in FEATURES_TO_RUN:
    feature_csv = FEATURE_CONFIGS[feature_name]["feature_csv"]
    assert feature_csv.exists(), f"Feature CSV not found: {feature_csv}"
    print(feature_name, "->", feature_csv)


In [ ]:
# 3 load the fixed split

# use the same speaker safe split for every feature set
SPLIT_CSV = SPLIT_ROOT / "FIN_split_70_30_by_language_grouped_by_safe_speaker.csv"

assert SPLIT_CSV.exists(), f"Split CSV not found: {SPLIT_CSV}"

split_df = pd.read_csv(SPLIT_CSV)

# confirm the columns needed for matching and leakage checks
required_split_cols = [
    "split",
    "language",
    "file_id",
    "label_name",
    "label",
    "speaker_id",
    "unique_audio_id",
]

missing = [c for c in required_split_cols if c not in split_df.columns]
assert len(missing) == 0, f"Missing split columns: {missing}"

print("Split shape:", split_df.shape)

# inspect class counts by split and language
display(
    split_df
    .groupby(["split", "language", "label_name"])
    .size()
    .reset_index(name="count")
)


In [ ]:
# 4 helper functions for matching acoustic row

def infer_language_from_dataset_or_path(dataset_name="", wav_path=""):
    """infer language from dataset or path text"""

    text = str(dataset_name) + " " + str(wav_path)

    if "Pitt" in text or "Pits" in text or "English" in text:
        return "English"

    if "Greek" in text or "DemCare" in text or "Dem@Care" in text:
        return "Greek"

    if "Mandarin" in text or "Chou" in text or "Chinese" in text:
        return "Mandarin"

    return "Unknown"


def normalize_file_id(value):
    """remove common audio and transcript suffixes"""

    value = str(value)

    value = value.replace("_patient.wav", "")
    value = value.replace("_patient", "")
    value = value.replace(".wav", "")
    value = value.replace(".cha", "")

    return value


def load_and_prepare_acoustic_csv(feature_csv, split_df):
    """
    load one acoustic csv and align it with the fixed split

    returns the feature matrix labels metadata and feature names
    """

    # load the selected acoustic table
    raw_df = pd.read_csv(feature_csv)

    print("\nRaw feature CSV:", feature_csv)
    print("Raw shape:", raw_df.shape)

    df = raw_df.copy()

    # create a normalized file id from the available path column
    if "file_id" in df.columns:
        df["file_id"] = df["file_id"].apply(normalize_file_id)

    elif "name" in df.columns:
        df["file_id"] = df["name"].apply(normalize_file_id)

    elif "wav_path" in df.columns:
        df["file_id"] = df["wav_path"].apply(
            lambda p: normalize_file_id(Path(str(p)).stem)
        )

    elif "file" in df.columns:
        df["file_id"] = df["file"].apply(
            lambda p: normalize_file_id(Path(str(p)).stem)
        )

    else:
        possible_path_cols = [
            c for c in df.columns
            if "file" in c.lower()
            or "path" in c.lower()
            or "name" in c.lower()
        ]
        print("Possible path or name columns:", possible_path_cols)
        raise ValueError(
            "Could not find file_id, name, wav_path, or file column."
        )

    # keep a consistent wav path column
    if "wav_path" not in df.columns:
        df["wav_path"] = ""

    # prepare the split metadata used as the matching reference
    split_meta = split_df[
        [
            "language",
            "file_id",
            "label_name",
            "label",
            "speaker_id",
            "unique_audio_id",
            "split",
        ]
    ].copy()

    split_meta["file_id"] = split_meta["file_id"].apply(normalize_file_id)

    # use saved language values or infer them from the path
    if "language" in df.columns:
        df["language"] = df["language"].astype(str)
    else:
        df["language"] = df["wav_path"].apply(
            lambda p: infer_language_from_dataset_or_path("", p)
        )

    print("\nAcoustic language counts before merge:")
    print(df["language"].value_counts())

    # match by language and file id first
    df_merged = df.merge(
        split_meta,
        on=["language", "file_id"],
        how="left",
        suffixes=("", "_split"),
    )

    print("\nAfter merge shape:", df_merged.shape)
    print(
        "Missing labels after language and file id merge:",
        df_merged["label_name"].isna().sum(),
    )

    # use file id alone only when it is unique in the split
    if df_merged["label_name"].isna().sum() > 0:
        unmatched = df_merged[df_merged["label_name"].isna()].copy()
        matched = df_merged[~df_merged["label_name"].isna()].copy()

        print(
            "\nTrying unique file id fallback for unmatched rows:",
            len(unmatched),
        )

        split_file_counts = (
            split_meta
            .groupby("file_id")
            .size()
            .reset_index(name="n")
        )

        unique_file_ids = split_file_counts[
            split_file_counts["n"] == 1
        ]["file_id"]

        split_meta_unique_file = split_meta[
            split_meta["file_id"].isin(unique_file_ids)
        ].copy()

        cols_to_drop = [
            "label_name",
            "label",
            "speaker_id",
            "unique_audio_id",
            "split",
        ]
        cols_to_drop = [
            c for c in cols_to_drop
            if c in unmatched.columns
        ]

        unmatched_base = unmatched.drop(columns=cols_to_drop)

        fallback = unmatched_base.merge(
            split_meta_unique_file,
            on="file_id",
            how="left",
            suffixes=("", "_fallback"),
        )

        # replace unknown language with the split language
        if "language_fallback" in fallback.columns:
            fallback["language"] = np.where(
                fallback["language"].astype(str).eq("Unknown"),
                fallback["language_fallback"].astype(str),
                fallback["language"].astype(str),
            )
            fallback = fallback.drop(columns=["language_fallback"])

        df_merged = pd.concat(
            [matched, fallback],
            ignore_index=True,
        )

    print(
        "\nFinal missing labels:",
        df_merged["label_name"].isna().sum(),
    )

    if df_merged["label_name"].isna().sum() > 0:
        display(
            df_merged[
                df_merged["label_name"].isna()
            ][["file_id", "language", "wav_path"]].head(50)
        )

    assert df_merged["label_name"].notna().all(), (
        "Some acoustic rows could not be matched to the split."
    )

    # finalize labels and unique matching keys
    df = df_merged.copy()

    df["label"] = df["label"].astype(int)
    df["label_name"] = df["label_name"].astype(str)
    df["speaker_id"] = df["speaker_id"].astype(str)
    df["unique_audio_id"] = df["unique_audio_id"].astype(str)
    df["key"] = df["unique_audio_id"].astype(str)

    # remove duplicate recordings before creating the matrix
    if df["key"].duplicated().any():
        print("Duplicate keys found:", df["key"].duplicated().sum())
        df = (
            df
            .drop_duplicates(subset="key", keep="first")
            .reset_index(drop=True)
        )
        print("After duplicate removal:", df.shape)

    # exclude metadata from the model input
    metadata_cols = {
        "dataset",
        "language",
        "file_id",
        "label",
        "label_name",
        "speaker_id",
        "unique_audio_id",
        "key",
        "wav_path",
        "file",
        "name",
        "start",
        "end",
        "duration",
        "status",
        "error",
    }

    candidate_feature_cols = [
        c for c in df.columns
        if c not in metadata_cols
    ]

    # keep columns with at least one numeric value
    numeric_feature_cols = []

    for col in candidate_feature_cols:
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().sum() > 0:
            df[col] = converted
            numeric_feature_cols.append(col)

    assert len(numeric_feature_cols) > 0, (
        "No numeric feature columns found."
    )

    X_all = df[numeric_feature_cols].values.astype(np.float32)
    y_all = df["label"].values.astype(int)

    # keep metadata aligned row for row with the feature matrix
    metadata_output_cols = [
        "language",
        "file_id",
        "label",
        "label_name",
        "speaker_id",
        "unique_audio_id",
        "key",
        "wav_path",
    ]

    if "dataset" in df.columns:
        metadata_output_cols.insert(0, "dataset")

    meta_all = df[metadata_output_cols].copy()

    print("Numeric feature columns:", len(numeric_feature_cols))
    print("X_all:", X_all.shape)
    print("y_all:", y_all.shape)

    return {
        "X": X_all,
        "y": y_all,
        "meta": meta_all,
        "feature_columns": numeric_feature_cols,
        "prepared_df": df,
    }


def match_acoustic_to_split(X, y, meta, split_df, language):
    """create one language specific train and test dataset"""

    meta = meta.copy()

    language_split = split_df[
        split_df["language"] == language
    ].copy()

    train_keys = set(
        language_split[
            language_split["split"] == "train"
        ]["unique_audio_id"].astype(str)
    )

    test_keys = set(
        language_split[
            language_split["split"] == "test"
        ]["unique_audio_id"].astype(str)
    )

    # confirm the fixed split has no recording overlap
    assert len(train_keys & test_keys) == 0, (
        f"Train and test overlap for {language}"
    )

    train_mask = meta["key"].astype(str).isin(train_keys).values
    test_mask = meta["key"].astype(str).isin(test_keys).values

    X_train = X[train_mask]
    y_train = y[train_mask]
    meta_train = meta[train_mask].copy()

    X_test = X[test_mask]
    y_test = y[test_mask]
    meta_test = meta[test_mask].copy()

    print("\n" + "=" * 80)
    print("Language:", language)
    print("Expected train:", len(train_keys))
    print("Found train:", len(meta_train))
    print("Missing train:", len(train_keys) - len(meta_train))
    print("Expected test:", len(test_keys))
    print("Found test:", len(meta_test))
    print("Missing test:", len(test_keys) - len(meta_test))
    print("Train labels:", np.bincount(y_train) if len(y_train) else [])
    print("Test labels:", np.bincount(y_test) if len(y_test) else [])
    print("=" * 80)

    return {
        "X_train": X_train.astype(np.float32),
        "y_train": y_train.astype(int),
        "meta_train": meta_train.reset_index(drop=True),
        "X_test": X_test.astype(np.float32),
        "y_test": y_test.astype(int),
        "meta_test": meta_test.reset_index(drop=True),
    }


In [ ]:
# 5 define models and evaluation


def make_models(use_pca=True, pca_components=0.95, random_state=42):
    """create fresh model pipelines"""

    # imputation and scaling are fitted on training data only
    scaled_steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]

    # optionally retain the requested proportion of variance
    if use_pca:
        scaled_steps.append(
            (
                "pca",
                PCA(
                    n_components=pca_components,
                    random_state=random_state,
                ),
            )
        )

    models = {}

    # nonlinear svm
    models["SVM_RBF"] = Pipeline(
        scaled_steps
        + [
            (
                "clf",
                SVC(
                    kernel="rbf",
                    C=1.0,
                    gamma="scale",
                    class_weight="balanced",
                    probability=True,
                    random_state=random_state,
                ),
            )
        ]
    )

    # linear svm
    models["SVM_Linear"] = Pipeline(
        scaled_steps
        + [
            (
                "clf",
                SVC(
                    kernel="linear",
                    C=1.0,
                    class_weight="balanced",
                    probability=True,
                    random_state=random_state,
                ),
            )
        ]
    )

    # tree models use imputation without scaling
    models["RandomForest"] = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=500,
                    max_depth=None,
                    min_samples_leaf=2,
                    class_weight="balanced",
                    random_state=random_state,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    # distance weighted nearest neighbours
    models["KNN"] = Pipeline(
        scaled_steps
        + [
            (
                "clf",
                KNeighborsClassifier(
                    n_neighbors=5,
                    weights="distance",
                ),
            )
        ]
    )

    # gradient boosted trees
    models["XGBoost"] = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "clf",
                XGBClassifier(
                    n_estimators=300,
                    max_depth=3,
                    learning_rate=0.03,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=random_state,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    # neural network with early stopping
    models["MLP"] = Pipeline(
        scaled_steps
        + [
            (
                "clf",
                MLPClassifier(
                    hidden_layer_sizes=(128, 64),
                    activation="relu",
                    alpha=1e-3,
                    learning_rate_init=1e-3,
                    max_iter=1000,
                    early_stopping=True,
                    random_state=random_state,
                ),
            )
        ]
    )

    return models


def get_positive_scores(model, X_test):
    """return a continuous score for mci label 1"""

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
        if proba.shape[1] == 2:
            return proba[:, 1]

    if hasattr(model, "decision_function"):
        return model.decision_function(X_test)

    return None


def evaluate_train_test(
    experiment_name,
    feature_name,
    language,
    X_train,
    y_train,
    X_test,
    y_test,
    use_pca=True,
    pca_components=0.95,
    random_state=42,
):
    """fit on train data and evaluate on the fixed test data"""

    rows = []

    print("\n" + "=" * 80)
    print("Experiment:", experiment_name)
    print("Feature:", feature_name)
    print("Language:", language)
    print("X_train:", X_train.shape)
    print("X_test:", X_test.shape)
    print("Train labels:", np.bincount(y_train) if len(y_train) else [])
    print("Test labels:", np.bincount(y_test) if len(y_test) else [])
    print("PCA:", use_pca, pca_components)
    print("=" * 80)

    # return a structured error row for unusable data
    if len(y_train) == 0 or len(y_test) == 0:
        return pd.DataFrame(
            [
                {
                    "experiment": experiment_name,
                    "feature_set": feature_name,
                    "language": language,
                    "model": "",
                    "use_pca": use_pca,
                    "pca_components": pca_components,
                    "n_train": len(y_train),
                    "n_test": len(y_test),
                    "n_features": (
                        X_train.shape[1]
                        if len(X_train)
                        else np.nan
                    ),
                    "accuracy": np.nan,
                    "balanced_accuracy": np.nan,
                    "precision": np.nan,
                    "recall": np.nan,
                    "f1": np.nan,
                    "roc_auc": np.nan,
                    "error": "empty train or test set",
                }
            ]
        )

    if len(np.unique(y_train)) < 2:
        return pd.DataFrame(
            [
                {
                    "experiment": experiment_name,
                    "feature_set": feature_name,
                    "language": language,
                    "model": "",
                    "use_pca": use_pca,
                    "pca_components": pca_components,
                    "n_train": len(y_train),
                    "n_test": len(y_test),
                    "n_features": X_train.shape[1],
                    "accuracy": np.nan,
                    "balanced_accuracy": np.nan,
                    "precision": np.nan,
                    "recall": np.nan,
                    "f1": np.nan,
                    "roc_auc": np.nan,
                    "error": "training set has fewer than two classes",
                }
            ]
        )

    models = make_models(
        use_pca=use_pca,
        pca_components=pca_components,
        random_state=random_state,
    )

    # isolate model failures so the experiment grid continues
    for model_name, model in models.items():
        try:
            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)
            y_score = get_positive_scores(model, X_test)

            if y_score is not None and len(np.unique(y_test)) == 2:
                roc_auc = roc_auc_score(y_test, y_score)
            else:
                roc_auc = np.nan

            report = classification_report(
                y_test,
                y_pred,
                labels=[0, 1],
                target_names=["Control", "MCI"],
                output_dict=True,
                zero_division=0,
            )

            rows.append(
                {
                    "experiment": experiment_name,
                    "feature_set": feature_name,
                    "language": language,
                    "model": model_name,
                    "use_pca": use_pca,
                    "pca_components": pca_components,
                    "n_train": len(y_train),
                    "n_test": len(y_test),
                    "n_features": X_train.shape[1],
                    "accuracy": accuracy_score(y_test, y_pred),
                    "balanced_accuracy": balanced_accuracy_score(
                        y_test,
                        y_pred,
                    ),
                    "precision": precision_score(
                        y_test,
                        y_pred,
                        pos_label=1,
                        zero_division=0,
                    ),
                    "recall": recall_score(
                        y_test,
                        y_pred,
                        pos_label=1,
                        zero_division=0,
                    ),
                    "f1": f1_score(
                        y_test,
                        y_pred,
                        pos_label=1,
                        zero_division=0,
                    ),
                    "roc_auc": roc_auc,
                    "precision_macro": report["macro avg"]["precision"],
                    "recall_macro": report["macro avg"]["recall"],
                    "f1_macro": report["macro avg"]["f1-score"],
                    "precision_weighted": report["weighted avg"]["precision"],
                    "recall_weighted": report["weighted avg"]["recall"],
                    "f1_weighted": report["weighted avg"]["f1-score"],
                    "precision_control": report["Control"]["precision"],
                    "recall_control": report["Control"]["recall"],
                    "f1_control": report["Control"]["f1-score"],
                    "precision_mci": report["MCI"]["precision"],
                    "recall_mci": report["MCI"]["recall"],
                    "f1_mci": report["MCI"]["f1-score"],
                    "confusion_matrix": str(
                        confusion_matrix(
                            y_test,
                            y_pred,
                            labels=[0, 1],
                        ).tolist()
                    ),
                    "error": "",
                }
            )

        except Exception as exc:
            rows.append(
                {
                    "experiment": experiment_name,
                    "feature_set": feature_name,
                    "language": language,
                    "model": model_name,
                    "use_pca": use_pca,
                    "pca_components": pca_components,
                    "n_train": len(y_train),
                    "n_test": len(y_test),
                    "n_features": (
                        X_train.shape[1]
                        if len(X_train)
                        else np.nan
                    ),
                    "accuracy": np.nan,
                    "balanced_accuracy": np.nan,
                    "precision": np.nan,
                    "recall": np.nan,
                    "f1": np.nan,
                    "roc_auc": np.nan,
                    "precision_macro": np.nan,
                    "recall_macro": np.nan,
                    "f1_macro": np.nan,
                    "precision_weighted": np.nan,
                    "recall_weighted": np.nan,
                    "f1_weighted": np.nan,
                    "precision_control": np.nan,
                    "recall_control": np.nan,
                    "f1_control": np.nan,
                    "precision_mci": np.nan,
                    "recall_mci": np.nan,
                    "f1_mci": np.nan,
                    "confusion_matrix": "",
                    "error": str(exc),
                }
            )

    return pd.DataFrame(rows)


In [ ]:
# 6 build multilingual and cross-lingual datasets

LANGUAGES = ["English", "Greek", "Mandarin"]

CROSS_LINGUAL_SETUPS = [
    {
        "train_languages": ["English", "Greek"],
        "test_language": "Mandarin",
        "setup_name": "train_English_Greek_test_Mandarin",
    },
    {
        "train_languages": ["English", "Mandarin"],
        "test_language": "Greek",
        "setup_name": "train_English_Mandarin_test_Greek",
    },
    {
        "train_languages": ["Greek", "Mandarin"],
        "test_language": "English",
        "setup_name": "train_Greek_Mandarin_test_English",
    },
]


def build_multilingual_train_test(
    mono_data,
    test_language,
    languages,
):
    """train on all language train splits and test one language"""

    X_train = np.vstack(
        [
            mono_data[lang]["X_train"]
            for lang in languages
        ]
    ).astype(np.float32)

    y_train = np.concatenate(
        [
            mono_data[lang]["y_train"]
            for lang in languages
        ]
    ).astype(int)

    meta_train = pd.concat(
        [
            mono_data[lang]["meta_train"].copy()
            for lang in languages
        ],
        ignore_index=True,
    )

    X_test = mono_data[test_language]["X_test"].astype(np.float32)
    y_test = mono_data[test_language]["y_test"].astype(int)
    meta_test = mono_data[test_language]["meta_test"].copy()

    # check exact recording overlap
    overlap = (
        set(meta_train["key"].astype(str))
        & set(meta_test["key"].astype(str))
    )

    assert len(overlap) == 0, (
        f"Leakage detected for {test_language}: "
        f"{len(overlap)} overlapping keys."
    )

    return (
        X_train,
        y_train,
        X_test,
        y_test,
        meta_train,
        meta_test,
    )


def build_crosslingual_train_test(
    mono_data,
    train_languages,
    test_language,
):
    """train on two languages and test the held out language"""

    X_train = np.vstack(
        [
            mono_data[lang]["X_train"]
            for lang in train_languages
        ]
    ).astype(np.float32)

    y_train = np.concatenate(
        [
            mono_data[lang]["y_train"]
            for lang in train_languages
        ]
    ).astype(int)

    meta_train = pd.concat(
        [
            mono_data[lang]["meta_train"].copy()
            for lang in train_languages
        ],
        ignore_index=True,
    )

    X_test = mono_data[test_language]["X_test"].astype(np.float32)
    y_test = mono_data[test_language]["y_test"].astype(int)
    meta_test = mono_data[test_language]["meta_test"].copy()

    # confirm the held out language is absent from training
    assert test_language not in set(
        meta_train["language"].astype(str)
    ), f"Test language {test_language} appears in training data."

    # check exact recording overlap
    overlap = (
        set(meta_train["key"].astype(str))
        & set(meta_test["key"].astype(str))
    )

    assert len(overlap) == 0, (
        f"Leakage detected for {test_language}: "
        f"{len(overlap)} overlapping keys."
    )

    return (
        X_train,
        y_train,
        X_test,
        y_test,
        meta_train,
        meta_test,
    )


In [ ]:

# 7 run one complete feature experiment

def run_feature_experiments(
    feature_name,
    feature_csv,
    result_prefix,
    split_df,
    languages,
):
    """
    run monolingual multilingual and cross-lingual experiments
    for one acoustic feature table
    """

    print("\n" + "#" * 90)
    print("Running feature set:", feature_name)
    print("#" * 90)

    # load the selected data once
    prepared = load_and_prepare_acoustic_csv(
        feature_csv=feature_csv,
        split_df=split_df,
    )

    X_all = prepared["X"]
    y_all = prepared["y"]
    meta_all = prepared["meta"]
    numeric_feature_cols = prepared["feature_columns"]

    # create fixed train and test subsets for each language
    mono_data = {}

    for language in languages:
        mono_data[language] = match_acoustic_to_split(
            X=X_all,
            y=y_all,
            meta=meta_all,
            split_df=split_df,
            language=language,
        )

    # run monolingual experiments
    monolingual_results = []

    for language in languages:
        obj = mono_data[language]

        for use_pca, pca_components, pca_name in [
            (True, 0.95, "pca"),
            (False, None, "no_pca"),
        ]:
            res = evaluate_train_test(
                experiment_name=(
                    f"monolingual_{language}_"
                    f"{feature_name}_{pca_name}"
                ),
                feature_name=feature_name,
                language=language,
                X_train=obj["X_train"],
                y_train=obj["y_train"],
                X_test=obj["X_test"],
                y_test=obj["y_test"],
                use_pca=use_pca,
                pca_components=pca_components,
                random_state=42,
            )

            res["setting"] = "monolingual"
            res["train_languages"] = language
            res["test_language"] = language

            monolingual_results.append(res)

    monolingual_results_df = pd.concat(
        monolingual_results,
        ignore_index=True,
    )

    monolingual_path = (
        PROJECT
        / f"{result_prefix}_model_results_monolingual.csv"
    )

    monolingual_results_df.to_csv(
        monolingual_path,
        index=False,
    )

    # run multilingual experiments
    multilingual_results = []

    for test_language in languages:
        (
            X_train,
            y_train,
            X_test,
            y_test,
            meta_train,
            meta_test,
        ) = build_multilingual_train_test(
            mono_data=mono_data,
            test_language=test_language,
            languages=languages,
        )

        for use_pca, pca_components, pca_name in [
            (True, 0.95, "pca"),
            (False, None, "no_pca"),
        ]:
            res = evaluate_train_test(
                experiment_name=(
                    f"multilingual_train_all_test_"
                    f"{test_language}_{feature_name}_{pca_name}"
                ),
                feature_name=feature_name,
                language=test_language,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                use_pca=use_pca,
                pca_components=pca_components,
                random_state=42,
            )

            res["setting"] = "multilingual_train_all_languages"
            res["train_languages"] = "+".join(languages)
            res["test_language"] = test_language

            multilingual_results.append(res)

    multilingual_results_df = pd.concat(
        multilingual_results,
        ignore_index=True,
    )

    multilingual_path = (
        PROJECT
        / (
            f"{result_prefix}_model_results_"
            "multilingual_train_all_test_each_language.csv"
        )
    )

    multilingual_results_df.to_csv(
        multilingual_path,
        index=False,
    )

    # run cross-lingual experiments
    crosslingual_results = []

    for setup in CROSS_LINGUAL_SETUPS:
        train_languages = setup["train_languages"]
        test_language = setup["test_language"]
        setup_name = setup["setup_name"]

        (
            X_train,
            y_train,
            X_test,
            y_test,
            meta_train,
            meta_test,
        ) = build_crosslingual_train_test(
            mono_data=mono_data,
            train_languages=train_languages,
            test_language=test_language,
        )

        for use_pca, pca_components, pca_name in [
            (True, 0.95, "pca"),
            (False, None, "no_pca"),
        ]:
            res = evaluate_train_test(
                experiment_name=(
                    f"crosslingual_{setup_name}_"
                    f"{feature_name}_{pca_name}"
                ),
                feature_name=feature_name,
                language=test_language,
                X_train=X_train,
                y_train=y_train,
                X_test=X_test,
                y_test=y_test,
                use_pca=use_pca,
                pca_components=pca_components,
                random_state=42,
            )

            res["setting"] = "crosslingual_train_two_languages"
            res["train_languages"] = "+".join(train_languages)
            res["test_language"] = test_language
            res["setup_name"] = setup_name

            crosslingual_results.append(res)

    crosslingual_results_df = pd.concat(
        crosslingual_results,
        ignore_index=True,
    )

    crosslingual_path = (
        PROJECT
        / (
            f"{result_prefix}_model_results_"
            "crosslingual_train_two_test_heldout_language.csv"
        )
    )

    crosslingual_results_df.to_csv(
        crosslingual_path,
        index=False,
    )

    # combine every setting for this feature set
    all_results_df = pd.concat(
        [
            monolingual_results_df,
            multilingual_results_df,
            crosslingual_results_df,
        ],
        ignore_index=True,
    )

    all_results_path = (
        PROJECT
        / f"{result_prefix}_model_results_all_settings.csv"
    )

    all_results_df.to_csv(
        all_results_path,
        index=False,
    )

    print("\nSaved feature results:")
    print("Monolingual:", monolingual_path)
    print("Multilingual:", multilingual_path)
    print("Cross-lingual:", crosslingual_path)
    print("All settings:", all_results_path)
    print("Feature columns:", len(numeric_feature_cols))

    return {
        "feature_name": feature_name,
        "feature_csv": feature_csv,
        "feature_columns": numeric_feature_cols,
        "mono_data": mono_data,
        "monolingual_results": monolingual_results_df,
        "multilingual_results": multilingual_results_df,
        "crosslingual_results": crosslingual_results_df,
        "all_results": all_results_df,
        "paths": {
            "monolingual": monolingual_path,
            "multilingual": multilingual_path,
            "crosslingual": crosslingual_path,
            "all_results": all_results_path,
        },
    }


In [ ]:

# 8 run the selected feature sets

# keep all feature outputs in one dictionary
feature_runs = {}
combined_result_parts = []

for feature_name in FEATURES_TO_RUN:
    config = FEATURE_CONFIGS[feature_name]

    run_output = run_feature_experiments(
        feature_name=feature_name,
        feature_csv=config["feature_csv"],
        result_prefix=config["result_prefix"],
        split_df=split_df,
        languages=LANGUAGES,
    )

    feature_runs[feature_name] = run_output
    combined_result_parts.append(run_output["all_results"])

# combine egemaps and compare results for direct comparison
combined_results_df = pd.concat(
    combined_result_parts,
    ignore_index=True,
)

COMBINED_RESULTS_PATH = (
    PROJECT
    / "interpretable_egemaps_compare2016_model_results_all_settings.csv"
)

combined_results_df.to_csv(
    COMBINED_RESULTS_PATH,
    index=False,
)

print("\nSaved combined results:", COMBINED_RESULTS_PATH)
print("Combined shape:", combined_results_df.shape)

display(
    combined_results_df.sort_values(
        by=[
            "balanced_accuracy",
            "f1",
            "roc_auc",
            "accuracy",
        ],
        ascending=False,
    ).head(50)
)


In [ ]:

# 9 save comparison summaries

# rank all models across both feature sets
best_overall = (
    combined_results_df
    .sort_values(
        by=[
            "balanced_accuracy",
            "f1",
            "roc_auc",
            "accuracy",
        ],
        ascending=False,
    )
    .head(50)
    .reset_index(drop=True)
)

# keep the leading models for each feature and experiment setting
best_by_feature_setting = (
    combined_results_df
    .sort_values(
        by=[
            "balanced_accuracy",
            "f1",
            "roc_auc",
            "accuracy",
        ],
        ascending=False,
    )
    .groupby(["feature_set", "setting"])
    .head(10)
    .reset_index(drop=True)
)

# keep the leading models for each target language
best_by_feature_language = (
    combined_results_df
    .sort_values(
        by=[
            "balanced_accuracy",
            "f1",
            "roc_auc",
            "accuracy",
        ],
        ascending=False,
    )
    .groupby(["feature_set", "setting", "test_language"])
    .head(5)
    .reset_index(drop=True)
)

BEST_OVERALL_PATH = (
    PROJECT
    / "interpretable_egemaps_compare2016_best_overall.csv"
)

BEST_BY_FEATURE_SETTING_PATH = (
    PROJECT
    / "interpretable_egemaps_compare2016_best_by_feature_setting.csv"
)

BEST_BY_FEATURE_LANGUAGE_PATH = (
    PROJECT
    / "interpretable_egemaps_compare2016_best_by_feature_language.csv"
)

best_overall.to_csv(BEST_OVERALL_PATH, index=False)
best_by_feature_setting.to_csv(
    BEST_BY_FEATURE_SETTING_PATH,
    index=False,
)
best_by_feature_language.to_csv(
    BEST_BY_FEATURE_LANGUAGE_PATH,
    index=False,
)

print("Saved:", BEST_OVERALL_PATH)
print("Saved:", BEST_BY_FEATURE_SETTING_PATH)
print("Saved:", BEST_BY_FEATURE_LANGUAGE_PATH)

display(best_by_feature_setting)


In [ ]:
# 10 final summary

print("Completed feature sets:", FEATURES_TO_RUN)
print("\nCombined results:")
print(COMBINED_RESULTS_PATH)

print("\nComparison summaries:")
print(BEST_OVERALL_PATH)
print(BEST_BY_FEATURE_SETTING_PATH)
print(BEST_BY_FEATURE_LANGUAGE_PATH)

print("\nIndividual feature outputs:")
for feature_name, output in feature_runs.items():
    print("\n", feature_name)
    for result_type, path in output["paths"].items():
        print(result_type, "->", path)
